# Generic-PDK grating coupler: fiber to chip

This version uses GDSFactory's built-in elliptical grating coupler and generic layer stack. Its Gaussian source and field monitors are derived from the component geometry so the setup stays aligned when the PCell parameters change.

In [ ]:
from math import cos, radians, sin

import gdsfactory as gf

from gsim import fdtd

FIBER_ANGLE_DEG = 15.0
FIBER_HEIGHT_UM = 1.05
Z_BOUNDS = (-1.0, 1.55)

## Build the generic-PDK component

In [ ]:
pdk = gf.gpdk.PDK
pdk.activate()
component = gf.components.grating_coupler_elliptical(fiber_angle=FIBER_ANGLE_DEG)
component

## Derive the simulation geometry

The built-in component exposes a guided port `o1` and a vertical fiber port `o2`. We use `o2` to place a tilted Gaussian aperture, while preserving analytic source normalization at `o1`.

In [ ]:
fiber_port = component.ports["o2"]
fiber_x_um, fiber_y_um = map(float, fiber_port.center)
fiber_width_um = float(fiber_port.width)
fiber_angle_rad = radians(FIBER_ANGLE_DEG)
fiber_center_um = (fiber_x_um, fiber_y_um, FIBER_HEIGHT_UM)
fiber_size_um = (fiber_width_um, fiber_width_um, 0.0)
fiber_direction = (-sin(fiber_angle_rad), 0.0, -cos(fiber_angle_rad))

bbox = component.dbbox()
device_center_x_um = (bbox.left + bbox.right) / 2
device_center_y_um = (bbox.bottom + bbox.top) / 2
device_span_x_um = bbox.right - bbox.left
device_span_y_um = bbox.top - bbox.bottom
device_center_z_um = sum(Z_BOUNDS) / 2
device_span_z_um = Z_BOUNDS[1] - Z_BOUNDS[0]

In [ ]:
simulation = fdtd.Simulation(pdk=pdk)
simulation.materials(background="sio2", overrides={"si": 3.47, "sio2": 1.443})
simulation.geometry(component, mesh_size_nm=500, geometry_tolerance_nm=20)
simulation.source = fdtd.GaussianBeamSource(
    center_um=fiber_center_um,
    size_um=fiber_size_um,
    aperture_normal="-z",
    propagation_direction=fiber_direction,
    e_polarization=(0.0, 1.0, 0.0),
    focal_point_um=(fiber_x_um, fiber_y_um, 0.22),
    waist_radius_um=fiber_width_um / 2,
    refractive_index=1.443,
    wavelength_um=1.554,
    wavelength_span_um=0.1,
    num_wavelengths=101,
)
simulation.monitors.add_plane(
    name="cross",
    center_um=(device_center_x_um, 0.0, device_center_z_um),
    size_um=(device_span_x_um, 0.0, device_span_z_um),
    normal="+y",
    flux=False,
    heatmap=fdtd.Heatmap(quantity="abs_e", wavelengths_um=[1.554]),
)
simulation.monitors.add_plane(
    name="top_down",
    center_um=(device_center_x_um, device_center_y_um, 0.075),
    size_um=(device_span_x_um, device_span_y_um, 0.0),
    normal="+z",
    flux=False,
    heatmap=fdtd.Heatmap(quantity="abs_e", wavelengths_um=[1.554]),
)
simulation.monitors.add_plane(
    name="src_down",
    center_um=(fiber_x_um, fiber_y_um, 0.4),
    size_um=fiber_size_um,
    normal="-z",
    flux=True,
)
simulation.domain(padding_um=0.5, z_bounds=Z_BOUNDS)
simulation.solver(cell_size_nm=50, energy_decay_fraction=1e-5)

In [ ]:
simulation.plot_3d()

## Run and normalize

The guided power at `o1` is divided by the analytic incident Gaussian power. Low-power spectral tails are masked by the normalization helper.

In [ ]:
result = simulation.run(check_cache=True)

In [ ]:
coupling = fdtd.gaussian_coupling_efficiency(result, simulation.source, port="o1")
coupling.plot_plotly()

## Inspect the fields

In [ ]:
result.monitors["cross"].plot_heatmap(
    wavelength_um=1.554, quantity="abs_e", cmap="magma"
)

In [ ]:
result.monitors["top_down"].plot_heatmap(
    wavelength_um=1.554, quantity="abs_e", cmap="magma"
)

The `src_down` monitor is retained as a flux diagnostic; reflected light makes its net flux unsuitable as the incident-power reference.